# Feature Selection using Ordinary Least Squares

## 1 Ordinary Least Squares (OLS)

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [2]:
RANDOM_STATE = 42

# Configure NumPy.
rng = np.random.default_rng(seed=RANDOM_STATE)
np.set_printoptions(linewidth=100)

### 1.1 Introduction

#### What is Ordinary Least Squares?

- **OLS is a statistical method** used to estimate the parameters of a linear regression model by finding the line or hyperplane that best fits the data.
- **Ordinary Least Squares** is an **alternate method** of performing Linear Regression **without Gradient Descent** Algorithm.
- MSE is used as cost function in Gradient Descent Algorithm where as **Sum of Squared Errors** is used as **cost function in OLS**.

#### Benefits of Ordinary Least Squares

- Ordinary Least Squares helps in **Feature selection**.

### 1.2 `statsmodels` lib

#### API

```python
import statsmodels.api as sm


model = sm.OLS(y, X)
model.fit()
```

#### Why add Constant manually?

Before adding constant:

$$
\large
\begin{aligned}
\hat{y}_1\;&=\;w_{10}\;+\;w_{11} \cdot x_{11}\;+\;w_{12} \cdot x_{12}\;+\;\ldots\;+\;w_{1d} \cdot x_{1d} \\[5pt]
\hat{y}_2\;&=\;w_{20}\;+\;w_{21} \cdot x_{21}\;+\;w_{22} \cdot x_{22}\;+\;\ldots\;+\;w_{2d} \cdot x_{2d} \\[5pt]
           &\vdots \\[5pt]
\hat{y}_n\;&=\;w_{n0}\;+\;w_{n1} \cdot x_{n1}\;+\;w_{n2} \cdot x_{n2}\;+\;\ldots\;+\;w_{nd} \cdot x_{nd}
\end{aligned}
$$

- In the beginning, shape of Features matrix is $(n,\,d)$ i.e., $n$ number of observations and $d$ number of features.
- The shape of the weight vector is $(1,\,d+1)$ i.e., $1$ row and $d+1$ columns.
- $d+1$ columns because, $d$ number of features plus $1$ for Bias or Intercept.
- To achieve dot product between weight and features, feature matrix must be of shape $(d+1,\,n)$.
- Criteria for dot product is: $(1,\,d+1)\;\times\;(d+1,\,n)$
- Hence **column of ones** (dummy feature) is added into features matrix to facilitate dot product.

After adding constant:

$$
\large
\begin{aligned}
\hat{y}_1\;&=\;w_{10} \cdot 1\;+\;w_{11} \cdot x_{11}\;+\;w_{12} \cdot x_{12}\;+\;\ldots\;+\;w_{1d} \cdot x_{1d} \\[5pt]
\hat{y}_2\;&=\;w_{20} \cdot 1\;+\;w_{21} \cdot x_{21}\;+\;w_{22} \cdot x_{22}\;+\;\ldots\;+\;w_{2d} \cdot x_{2d} \\[5pt]
           &\vdots \\[5pt]
\hat{y}_n\;&=\;w_{n0} \cdot 1\;+\;w_{n1} \cdot x_{n1}\;+\;w_{n2} \cdot x_{n2}\;+\;\ldots\;+\;w_{nd} \cdot x_{nd}
\end{aligned}
$$

> **Note**:
> 
> scikit-learn library add column of ones automatically by default using `fit_intercept=True` parameter.

## 2 Feature Selection with OLS

### 2.1 Hypothesis Tests in OLS

#### F-test

F-test to check if Linear Regression is possible on the given dataset.

#### T-test

T-test to check the usefulness of individual features.

### 2.2 Example

#### Import Dataset

In [3]:
df = pd.read_csv("data/cars24-car-price-scaled.csv")

display(df.head())
row_count, col_count = df.shape
print(f"Dataset has {row_count:,} rows and {col_count} columns.")

,selling_price,year,km_driven,mileage,engine,max_power,age,make,model,Individual,Trustmark Dealer,Diesel,Electric,LPG,Petrol,Manual,5,>5
0,-1.111046,-0.801317,1.195828,0.045745,-1.310754,-1.157780,0.801317,-0.433854,-1.125683,1.248892,-0.098382,-0.985275,-0.020095,-0.056917,1.024622,0.495818,0.444503,-0.424728
1,-0.223944,0.450030,-0.737872,-0.140402,-0.537456,-0.360203,-0.450030,-0.327501,-0.333227,1.248892,-0.098382,-0.985275,-0.020095,-0.056917,1.024622,0.495818,0.444503,-0.424728
2,-0.915058,-1.426990,0.035608,-0.582501,-0.537456,-0.404885,1.426990,-0.327501,-0.789807,1.248892,-0.098382,-0.985275,-0.020095,-0.056917,1.024622,0.495818,0.444503,-0.424728
3,-0.892365,-0.801317,-0.409143,0.329620,-0.921213,-0.693085,0.801317,-0.433854,-0.905265,1.248892,-0.098382,-0.985275,-0.020095,-0.056917,1.024622,0.495818,0.444503,-0.424728
4,-0.182683,0.137194,-0.544502,0.760085,0.042999,0.010435,-0.137194,-0.246579,-0.013096,-0.800710,-0.098382,1.014945,-0.020095,-0.056917,-0.975970,0.495818,0.444503,-0.424728


Dataset has 19,820 rows and 18 columns.


#### Training Setup

##### Split Features and Target

In [4]:
X = df.drop(columns="selling_price")
y = df["selling_price"]

X.shape, y.shape

((19820, 17), (19820,))

##### Train Test Split

In [5]:
X_train_wi, X_test_wi, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train_wi.shape, X_test_wi.shape, y_train.shape, y_test.shape

((15856, 17), (3964, 17), (15856,), (3964,))

##### Add Intercept Manually

In [6]:
X_train = sm.add_constant(X_train_wi)
X_train.shape

(15856, 18)

In [7]:
X_test = sm.add_constant(X_test_wi)
X_test.shape

(3964, 18)

#### Training OLS model

In [8]:
model = sm.OLS(y_train, X_train)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:          selling_price   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.941
Method:                 Least Squares   F-statistic:                 1.500e+04
Date:                Wed, 14 Oct 2026   Prob (F-statistic):               0.00
Time:                        19:42:37   Log-Likelihood:                 67.640
No. Observations:               15856   AIC:                            -99.28
Df Residuals:                   15838   BIC:                             38.80
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -0.0003      0.002  